# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fayrouzhassan2000/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Importing The Data

In [63]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
    CREATE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    )
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

In [64]:
features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        AVG(gsc_avg_position) AS gsc_avg_position,
        SUM(ga4_sessions) AS ga4_sessions
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [65]:
from datasets import load_dataset

content_ds = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_content",
    split="train"
)

content_df = content_ds.to_pandas()

In [66]:
content_df = content_df[
    [
        "client_hash_id",
        "content_hash_id",
        "content_type"
    ]
]

In [67]:
feature_df = features.merge(
    content_df,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

In [68]:
feature_df.head()

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,content_type
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,7.209549,1.0,keyword article
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,2.987198,0.0,keyword article
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,6.724039,3.0,keyword article
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,7.244844,2.0,keyword article
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,42.0,0.0,14.432540,7.0,keyword article


In [69]:
feature_df.shape

(331437, 7)

In [70]:
april = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_impressions
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-04'
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [71]:
march = feature_df.merge(
    april,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

march["is_declining_label"] = (
    march["april_impressions"] < march["gsc_impressions"]
)

In [72]:
march.head()

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,content_type,april_impressions,is_declining_label
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,7.209549,1.0,keyword article,6787.0,False
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,2.987198,0.0,keyword article,405.0,True
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,6.724039,3.0,keyword article,8475.0,False
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,7.244844,2.0,keyword article,6091.0,False
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,42.0,0.0,14.432540,7.0,keyword article,67.0,False


In [73]:
# Drop april_impressions col to avoid data leakage

march.drop(columns=["april_impressions"], axis = 1, inplace=True)

In [74]:
march.isnull().sum()

,0
client_hash_id,0
content_hash_id,0
gsc_impressions,0
gsc_clicks,0
gsc_avg_position,154699
ga4_sessions,70700
content_type,0
is_declining_label,0


In [75]:
march.columns

Index(['client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks',
       'gsc_avg_position', 'ga4_sessions', 'content_type',
       'is_declining_label'],
      dtype='object')

In [76]:
df = march.copy()

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*


### Signal checks

I will audit two signals before encoding the baseline rule:

1. **Search volume (`gsc_impressions`)** — linked to FlyRank's volume-based quick-win logic.
2. **CTR relative to average position** — linked to FlyRank's CTR-fix logic.

For each signal, I will compare bucket-level decline rates and report the number of observations (`n`). The purpose is to check whether the signal has evidence supporting its use in the baseline rule.


In [77]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Signal 1: Impressions
df["impressions_bucket"] = pd.cut(
    df["gsc_impressions"],
    bins=[-1, 100, 500, 1000, 5000, float("inf")],
    labels=[
        "0-100",
        "101-500",
        "501-1k",
        "1k-5k",
        "5k+"
    ]
)

volume_check = (
    df.groupby("impressions_bucket", observed=False)
      .agg(
          n=("is_declining_label", "size"),
          decline_rate=("is_declining_label", "mean")
      )
      .reset_index()
)

volume_check["decline_rate"] *= 100

print(volume_check.to_string(index=False))


impressions_bucket      n  decline_rate
             0-100 230205     19.693751
           101-500  39356     65.740929
            501-1k  16841     66.997209
             1k-5k  31745     65.071665
               5k+  13290     66.350640


**Signal 1 Verdict: CONFIRMED**

Search volume is a useful signal in this dataset. Content with 0–100 impressions has a 19.69% decline rate, while all buckets above 100 impressions have decline rates around 65–67%.

This strong separation supports using search visibility as a prioritization signal. It is also linked to FlyRank's volume-based quick-win logic.


In [78]:
# Signal 2: CTR vs Position
df["ctr"] = np.where(
    df["gsc_impressions"] > 0,
    df["gsc_clicks"] / df["gsc_impressions"],
    np.nan
)

df["position_bucket"] = pd.cut(
    df["gsc_avg_position"],
    bins=[0, 3, 5, 10, 20, 50, float("inf")],
    labels=[
        "1-3",
        "4-5",
        "6-10",
        "11-20",
        "21-50",
        "50+"
    ]
)

position_check = (
    df.groupby("position_bucket", observed=False)
      .agg(
          n=("is_declining_label", "size"),
          median_ctr=("ctr", "median"),
          decline_rate=("is_declining_label", "mean")
      )
      .reset_index()
)

position_check["decline_rate"] *= 100

print(position_check.to_string(index=False))

position_bucket     n  median_ctr  decline_rate
            1-3 16144    0.000000     65.764371
            4-5 26593    0.000833     64.471853
           6-10 55395    0.000000     63.567109
          11-20 32203    0.000000     64.546781
          21-50 33288    0.000000     61.700913
            50+ 11681    0.000000     59.335673


  **Signal 2 Verdict: MIXED**

CTR relative to average position does not show a strong or consistent relationship with the decline label in this dataset. The decline rate varies only moderately across position buckets, and the median CTR is zero in most buckets.

Therefore, I will not use CTR relative to position in the baseline scoring rule.


## Encode One Rule

### Baseline rule

I will prioritize content with meaningful search visibility.

A content item is eligible for review when `gsc_impressions > 100`. The threshold is based on the signal audit: content above 100 impressions has a substantially higher observed decline rate than content with 0–100 impressions.

The rule uses:

* **Score:** `gsc_impressions` for eligible content; 0 otherwise.
* **Reason code:** `high_search_visibility` for eligible content.
* **Action label:** `REVIEW` for eligible content and `NO_ACTION` otherwise.

This is a simple hand-written baseline with an explicit threshold and no learned weights.


In [79]:
IMPRESSION_THRESHOLD = 100

df["score"] = np.where(
    df["gsc_impressions"] > IMPRESSION_THRESHOLD,
    df["gsc_impressions"],
    0
)

df["reason_code"] = np.where(
    df["gsc_impressions"] > IMPRESSION_THRESHOLD,
    "high_search_visibility",
    "no_flag"
)

df["action"] = np.where(
    df["gsc_impressions"] > IMPRESSION_THRESHOLD,
    "REVIEW",
    "NO_ACTION"
)

In [80]:
print(df[
    [
        "gsc_impressions",
        "score",
        "reason_code",
        "action"
    ]
].head(10))

   gsc_impressions   score             reason_code     action
0           6523.0  6523.0  high_search_visibility     REVIEW
1            453.0   453.0  high_search_visibility     REVIEW
2           5630.0  5630.0  high_search_visibility     REVIEW
3           4944.0  4944.0  high_search_visibility     REVIEW
4             42.0     0.0                 no_flag  NO_ACTION
5            429.0   429.0  high_search_visibility     REVIEW
6            223.0   223.0  high_search_visibility     REVIEW
7             96.0     0.0                 no_flag  NO_ACTION
8            314.0   314.0  high_search_visibility     REVIEW
9           7709.0  7709.0  high_search_visibility     REVIEW


In [81]:
print("\nAction counts:")
print(df["action"].value_counts())


Action counts:
action
NO_ACTION    230205
REVIEW       101232
Name: count, dtype: int64


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [82]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
ranked_queue = df.sort_values(
    "score",
    ascending=False
).copy()


In [83]:
ranked_queue[
    [
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "score",
        "reason_code",
        "action"
    ]
].head(10)

,client_hash_id,content_hash_id,gsc_impressions,score,reason_code,action
257834,client_e547b89c05043229,content_eadb33b5df496f4a,617124.0,617124.0,high_search_visibility,REVIEW
258729,client_e547b89c05043229,content_ec2e0346994fb5a5,245276.0,245276.0,high_search_visibility,REVIEW
270393,client_23a62021009f63c4,content_e8a52cf3d5988c07,244931.0,244931.0,high_search_visibility,REVIEW
92104,client_e547b89c05043229,content_0e03de7680314cd5,221310.0,221310.0,high_search_visibility,REVIEW
210720,client_23a62021009f63c4,content_44f34c0a90047651,212404.0,212404.0,high_search_visibility,REVIEW
271222,client_62f4a7e64f5e0096,content_7172a7fad43f0998,205867.0,205867.0,high_search_visibility,REVIEW
59171,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,205045.0,205045.0,high_search_visibility,REVIEW
92069,client_e547b89c05043229,content_8d7d99f109e19aa2,203497.0,203497.0,high_search_visibility,REVIEW
106107,client_62f4a7e64f5e0096,content_f107e54b10b43725,195997.0,195997.0,high_search_visibility,REVIEW
41276,client_23a62021009f63c4,content_36e53e9c707674fc,194579.0,194579.0,high_search_visibility,REVIEW


In [84]:
output_cols = [
    "client_hash_id",
    "content_hash_id",
    "score",
    "reason_code",
    "action"
]

ranked_queue[output_cols].to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Saved:", "work/outputs/baseline_action_score.csv")

Saved: work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [85]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = ranked_queue.head(20).copy()

top20[
    [
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "score",
        "reason_code",
        "action"
    ]
]

,client_hash_id,content_hash_id,gsc_impressions,score,reason_code,action
257834,client_e547b89c05043229,content_eadb33b5df496f4a,617124.0,617124.0,high_search_visibility,REVIEW
258729,client_e547b89c05043229,content_ec2e0346994fb5a5,245276.0,245276.0,high_search_visibility,REVIEW
270393,client_23a62021009f63c4,content_e8a52cf3d5988c07,244931.0,244931.0,high_search_visibility,REVIEW
92104,client_e547b89c05043229,content_0e03de7680314cd5,221310.0,221310.0,high_search_visibility,REVIEW
210720,client_23a62021009f63c4,content_44f34c0a90047651,212404.0,212404.0,high_search_visibility,REVIEW
271222,client_62f4a7e64f5e0096,content_7172a7fad43f0998,205867.0,205867.0,high_search_visibility,REVIEW
59171,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,205045.0,205045.0,high_search_visibility,REVIEW
92069,client_e547b89c05043229,content_8d7d99f109e19aa2,203497.0,203497.0,high_search_visibility,REVIEW
106107,client_62f4a7e64f5e0096,content_f107e54b10b43725,195997.0,195997.0,high_search_visibility,REVIEW
41276,client_23a62021009f63c4,content_36e53e9c707674fc,194579.0,194579.0,high_search_visibility,REVIEW


In [86]:
top20[
    [
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_sessions",
        "content_type",
        "score",
        "reason_code",
        "action"
    ]
]

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,content_type,score,reason_code,action
257834,client_e547b89c05043229,content_eadb33b5df496f4a,617124.0,5668.0,2.383011,2730.0,keyword article,617124.0,high_search_visibility,REVIEW
258729,client_e547b89c05043229,content_ec2e0346994fb5a5,245276.0,1480.0,2.854514,816.0,keyword article,245276.0,high_search_visibility,REVIEW
270393,client_23a62021009f63c4,content_e8a52cf3d5988c07,244931.0,669.0,15.008339,891.0,keyword article,244931.0,high_search_visibility,REVIEW
92104,client_e547b89c05043229,content_0e03de7680314cd5,221310.0,720.0,2.675217,465.0,keyword article,221310.0,high_search_visibility,REVIEW
210720,client_23a62021009f63c4,content_44f34c0a90047651,212404.0,24.0,7.346909,37.0,keyword article,212404.0,high_search_visibility,REVIEW
271222,client_62f4a7e64f5e0096,content_7172a7fad43f0998,205867.0,862.0,3.367835,NaN,keyword article,205867.0,high_search_visibility,REVIEW
59171,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,205045.0,2446.0,4.544203,NaN,keyword article,205045.0,high_search_visibility,REVIEW
92069,client_e547b89c05043229,content_8d7d99f109e19aa2,203497.0,289.0,2.563756,164.0,keyword article,203497.0,high_search_visibility,REVIEW
106107,client_62f4a7e64f5e0096,content_f107e54b10b43725,195997.0,996.0,3.186054,NaN,keyword article,195997.0,high_search_visibility,REVIEW
41276,client_23a62021009f63c4,content_36e53e9c707674fc,194579.0,242.0,32.766674,2603.0,keyword article,194579.0,high_search_visibility,REVIEW


### Top-20 review

I reviewed the top 20 items selected by the baseline rule. The rule prioritizes content with more than 100 search impressions, using impressions as the ranking score.

For each item, I record the action, why the rule selected it, and what could make the recommendation wrong. The review is intentionally skeptical because high search visibility alone does not prove that a content item needs intervention.


### 1. Content `content_eadb33b5df496f4a`

**Action:** REVIEW
**Why it's here:** It has very high search visibility with 617,124 impressions, making it a high-priority item under the volume-based rule.
**What would make it wrong:** The high visibility may already be producing healthy engagement, so impressions alone may not indicate an actionable content problem.

### 2. Content `content_ec2e0346994fb5a5`

**Action:** REVIEW
**Why it's here:** It has 245,276 impressions, placing it among the highest-visibility items in the queue.
**What would make it wrong:** Its strong search position and existing clicks may indicate that the content is already performing appropriately.

### 3. Content `content_e8a52cf3d5988c07`

**Action:** REVIEW
**Why it's here:** It has 244,931 impressions, so the rule considers it a high-visibility item worth reviewing.
**What would make it wrong:** Its average position is relatively weak at 15.0, so the issue may be ranking rather than a content problem that this rule can diagnose.

### 4. Content `content_0e03de7680314cd5`

**Action:** REVIEW
**Why it's here:** It has 221,310 impressions and therefore receives a high priority from the volume-based score.
**What would make it wrong:** The page already ranks around position 2.7, so high visibility may be expected and not necessarily indicate a problem.

### 5. Content `content_44f34c0a90047651`

**Action:** REVIEW
**Why it's here:** It has 212,404 impressions, giving it a high priority under the baseline rule.
**What would make it wrong:** It has only 24 clicks despite high impressions, but this could reflect the type of search queries it ranks for rather than a content issue.

### 6. Content `content_7172a7fad43f0998`

**Action:** REVIEW
**Why it's here:** It has 205,867 impressions and therefore qualifies as a high-visibility item.
**What would make it wrong:** GA4 sessions are unavailable, so there is less supporting evidence from analytics to confirm that intervention is useful.

### 7. Content `content_e7b5dd4dff461ad2`

**Action:** REVIEW
**Why it's here:** It has 205,045 impressions, making it one of the highest-volume items in the queue.
**What would make it wrong:** GA4 sessions are unavailable, and high impressions alone do not establish that the content needs improvement.

### 8. Content `content_8d7d99f109e19aa2`

**Action:** REVIEW
**Why it's here:** It has 203,497 impressions and is therefore prioritized by the volume rule.
**What would make it wrong:** Its average position is strong at about 2.6, so the high visibility may simply reflect successful ranking.

### 9. Content `content_f107e54b10b43725`

**Action:** REVIEW
**Why it's here:** It has 195,997 impressions, giving it a high priority score.
**What would make it wrong:** GA4 sessions are unavailable, so the rule lacks an additional engagement signal to support the review.

### 10. Content `content_36e53e9c707674fc`

**Action:** REVIEW
**Why it's here:** It has 194,579 impressions and therefore meets the rule's high-visibility condition.
**What would make it wrong:** Its average position is about 32.8, so high impressions may reflect broad visibility rather than an obvious content-quality opportunity.

### 11. Content `content_b99ea6861864dea5`

**Action:** REVIEW
**Why it's here:** It has 194,337 impressions and receives a high priority under the volume rule.
**What would make it wrong:** Its average position is already around 4.5, so the high visibility may be normal for a well-ranking page.

### 12. Content `content_4ffe18112a5642e3`

**Action:** REVIEW
**Why it's here:** It has 186,983 impressions, placing it high in the ranked queue.
**What would make it wrong:** Its strong average position of about 2.3 suggests that visibility may already be healthy.

### 13. Content `content_acbcc847f8996314`

**Action:** REVIEW
**Why it's here:** It has 170,808 impressions and therefore qualifies for review under the baseline rule.
**What would make it wrong:** GA4 sessions are unavailable, and the rule cannot distinguish between a healthy high-volume page and one that needs intervention.

### 14. Content `content_471d9cabce329a66`

**Action:** REVIEW
**Why it's here:** It has 164,885 impressions, making it a meaningful search-visibility candidate.
**What would make it wrong:** Its position is about 4.7, which may already represent healthy search performance.

### 15. Content `content_512dbad65bd5ade9`

**Action:** REVIEW
**Why it's here:** It has 154,358 impressions and receives a high priority score.
**What would make it wrong:** It has 2,506 clicks and a strong average position of about 3.0, suggesting that the page may already be performing well.

### 16. Content `content_987d251ee617d9c6`

**Action:** REVIEW
**Why it's here:** It has 152,806 impressions and therefore qualifies as a high-visibility item.
**What would make it wrong:** Its strong average position of about 2.8 may mean that there is no actionable issue despite the high volume.

### 17. Content `content_fd2117c2c6790e4b`

**Action:** REVIEW
**Why it's here:** It has 151,166 impressions and is prioritized by the rule.
**What would make it wrong:** Its average position is strong at about 3.4, while its relatively low GA4 sessions may not be directly actionable from this rule.

### 18. Content `content_82e35c4845e6c391`

**Action:** REVIEW
**Why it's here:** It has 143,907 impressions, which is enough to trigger the high-visibility rule.
**What would make it wrong:** Its average position is about 22.6, so the low ranking may explain the observed traffic opportunity better than a generic content-review recommendation.

### 19. Content `content_34a70fea29d15f24`

**Action:** REVIEW
**Why it's here:** It has 143,019 impressions and therefore receives a high priority score.
**What would make it wrong:** It has only 43 clicks despite strong visibility, but this could be caused by query intent or SERP behavior rather than a content issue.

### 20. Content `content_e241d6415ac9e534`

**Action:** REVIEW
**Why it's here:** It has 142,304 impressions and meets the rule's high-visibility condition.
**What would make it wrong:** Its average position is strong at about 3.3, so the rule may flag a page that is already performing well.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks

The baseline rule is intentionally simple and uses search impressions as its only scoring signal. Because of this, some high-volume items may be false positives.

The weakest picks are items where high impressions are accompanied by evidence that the content may already be performing well, such as a strong average search position and meaningful clicks. These cases show where the baseline rule can over-prioritize content based on volume alone.


In [87]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

weak_picks = top20[
    (top20["gsc_avg_position"] <= 5) &
    (top20["gsc_clicks"] > 500)
].copy()

weak_picks[
    [
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_sessions",
        "content_type",
        "score",
        "reason_code",
        "action"
    ]
]

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,content_type,score,reason_code,action
257834,client_e547b89c05043229,content_eadb33b5df496f4a,617124.0,5668.0,2.383011,2730.0,keyword article,617124.0,high_search_visibility,REVIEW
258729,client_e547b89c05043229,content_ec2e0346994fb5a5,245276.0,1480.0,2.854514,816.0,keyword article,245276.0,high_search_visibility,REVIEW
92104,client_e547b89c05043229,content_0e03de7680314cd5,221310.0,720.0,2.675217,465.0,keyword article,221310.0,high_search_visibility,REVIEW
271222,client_62f4a7e64f5e0096,content_7172a7fad43f0998,205867.0,862.0,3.367835,NaN,keyword article,205867.0,high_search_visibility,REVIEW
59171,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,205045.0,2446.0,4.544203,NaN,keyword article,205045.0,high_search_visibility,REVIEW
106107,client_62f4a7e64f5e0096,content_f107e54b10b43725,195997.0,996.0,3.186054,NaN,keyword article,195997.0,high_search_visibility,REVIEW
92099,client_e547b89c05043229,content_4ffe18112a5642e3,186983.0,586.0,2.331060,364.0,keyword article,186983.0,high_search_visibility,REVIEW
139333,client_73cda7b4e4f265ea,content_512dbad65bd5ade9,154358.0,2506.0,3.019798,713.0,keyword article,154358.0,high_search_visibility,REVIEW
285937,client_73cda7b4e4f265ea,content_987d251ee617d9c6,152806.0,940.0,2.823429,277.0,keyword article,152806.0,high_search_visibility,REVIEW


These weak picks illustrate an important limitation of the baseline. High impressions identify content with substantial search visibility, but they do not necessarily indicate that the content is declining or needs intervention.

For example, items with strong average positions and substantial click volume can be healthy pages that were selected only because they have high search visibility. A stronger future model should combine volume with additional evidence of deterioration rather than treating high volume as sufficient on its own.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.